In [ ]:
!pip install numpy<2
import numpy as np
import pandas as pd
import string
!pip install wandb
!wandb login
!pip install contractions
!wandb login --relogin

In [ ]:
lyrics_data = pd.read_csv("/content/drive/MyDrive/ML DATASET/spotify_dataset.csv", #Importing the lyrics dataset with a sample of 10000 and only includes columns such as 'song', 'text', and 'emotion'
                        usecols = ['song', 'text', 'emotion',],
                        nrows = 100000,)
sample_lyrics = lyrics_data.sample(n = 500000, random_state = 42 )#Create a sample size from the imported sample with a size of 10000 and a random_state of 42
sample_lyrics['emotion'].unique()
emotion_dict = {
    'sadness': 0,
    'joy': 1,
    'love': 2,
    'anger': 3,
    'fear': 4,
    'surprise' : 5
}

sample_lyrics['emotion'] = sample_lyrics['emotion'].map(emotion_dict)
sample_lyrics = sample_lyrics.dropna(subset=['emotion'])
sample_lyrics['emotion'] = sample_lyrics['emotion'].astype('int64')
sample_lyrics.isna().sum()

In [ ]:
sample_lyrics

In [ ]:
#Check for dtype
if sample_lyrics['emotion'].dtype == 'int64':
  print('All emotions are integers')
else:
  print('Not all emotions are integers')

In [ ]:
sample_lyrics['emotion'].value_counts()
emotion_class = sample_lyrics['emotion'].value_counts().sort_index()
emotion_class

Visualize Emotion Class Distributions

In [ ]:
emotion_names = ['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise']
emotion_counts = sample_lyrics['emotion'].value_counts().sort_index()
colors = ['tab:blue', 'tab:cyan', 'tab:orange', 'tab:red', 'tab:purple', 'tab:green']

import matplotlib.pyplot as plt
plt.bar(emotion_names, emotion_counts.values, color = colors)
plt.title('Emotions Class Distribution')
plt.xlabel('Emotions')
plt.ylabel('Value Counts')
plt.show()

In [ ]:
'''
Some articles to handle imabalance class:
https://medium.com/@nikviz/bert-handling-class-imbalance-in-language-models-7fe9ccc62cb6
https://www.geeksforgeeks.org/machine-learning/handling-imbalanced-data-for-classification/
https://github.com/scikit-learn-contrib/imbalanced-learn

'''



Preprocessing Text Data


In [ ]:
#Expand Contraction (I'd => I would)
import contractions
def contract_word(lyrics):
  return contractions.fix(lyrics)
sample_lyrics['clean_text'] = sample_lyrics['text'].apply(contract_word)

#Remove [Verse 1], [Verse 2], etc
import re
def remove_song_structure(lyrics):
  lyrics = re.sub(r"\[.*?\]", ' ', lyrics)
  lyrics = re.sub(r"\s+", ' ', lyrics)

  return lyrics
sample_lyrics['clean_text'] = sample_lyrics['text'].apply(lambda lyrics: remove_song_structure(lyrics))

#Remove punctation
punc = "#$%&\'()*+,-./:;<=>@[\\]^_`{|}~"
def remove_certain_punc(lyrics):
    return lyrics.translate(str.maketrans('', '', punc))
sample_lyrics['clean_text'] = sample_lyrics['clean_text'].apply(lambda lyrics: remove_certain_punc(lyrics))

#Removing of 10 frequent words
from collections import Counter
word_count = Counter()
for lyrics in sample_lyrics['clean_text']:
    for word in lyrics.split():
        word_count[word] += 1

word_count.most_common(10)

frequent_words = set(word for (word, wc) in word_count.most_common(10))

def rem_freq(lyrics):
  word_count = " ".join([word for word in lyrics.split() if word not in frequent_words])
  return word_count
sample_lyrics['clean_text'] = sample_lyrics['clean_text'].apply(lambda x: rem_freq(x))

#Removing rare/ odd words
weird_words = set(word for (word, wc) in word_count.most_common()[:-10:-1])
def rem_rare(lyrics):
  uncommon_words = " ".join([word for word in lyrics.split() if word not in weird_words])
  return uncommon_words
sample_lyrics['clean_text'] = sample_lyrics['clean_text'].apply(lambda x: rem_rare(x))
sample_lyrics

#Lemmatization of words THIS ONE WAS INSPIRED BY CODE DESIGN FROM https://www.hackersrealm.net/post/text-preprocessing-in-nlp-python
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem.wordnet import WordNetLemmatizer


lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V": wordnet.VERB, "J": wordnet.ADJ, "R": wordnet.ADV}

def lemmatize_words(text):
    # find pos tags
    pos_text = pos_tag(text.split())
    return " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_text])

sample_lyrics['clean_text'] = sample_lyrics['clean_text'].apply(lambda x: lemmatize_words(x))

def remove_empty_struture(lyrics):
  structure = ['Hook', 'Verse 1', 'Verse 2', 'Verse 3', 'Verse1', 'Verse2', 'Verse3', 'Interlude']
  for tags in structure:
    lyrics = lyrics.replace(tags, '')
  return lyrics
sample_lyrics['clean_text'] = sample_lyrics['clean_text'].apply(remove_empty_struture)

#Convert everything to lower string
sample_lyrics['clean_text'] = sample_lyrics['clean_text'].str.lower()

#Split the datasets into train, validation, test sets

In [ ]:
from sklearn.model_selection import train_test_split #Import tran, test, split functions from skitlearn
new_data = sample_lyrics.drop(['text', 'song'], axis = 1) #Drop irrelevant columns
X = new_data.drop(['emotion'], axis =1 ) # Create Input data only by dropping 'emotion' column
y = new_data['emotion'] #Creating output data for predictions
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state =42) #Splitting the lyrics data into 70% for train and 30% for test

#BERT tokenizer can only tokenize LIST OF STRING so I have to convert it
input_train = X_train['clean_text'].tolist() #Convert input train set into a list of strings containg PREPROCESSED LYRICS
input_test = X_test['clean_text'].tolist() #Convert input test set into a list of strings containing PREPROCESSED LYRICS


In [ ]:
!pip install transformers
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
version = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(version)
model = AutoModel.from_pretrained(version)

train_df = pd.DataFrame({'train_text': input_train,
                         'emotion labels': y_train})

test_df = pd.DataFrame({'test_text': input_test,
                        'emotion labels': y_test})

from datasets import Dataset, DatasetDict


lyrics_dataset =  DatasetDict({

    'train_set' : Dataset.from_pandas(train_df),
    'test_set' : Dataset.from_pandas(test_df)

})

#Write functions
def tokenize_train(row): #row parameter is the train_set[0], train_set[1], train_set[2],etc
  return tokenizer(row['train_text'], padding = True, truncation = True) #Return a tokenized train text along with other column/features (emotion labels, index)

tokenize_train_apache = lyrics_dataset['train_set'].map(tokenize_train, batched = True) #Map the function on the train dataset, put the lyrics text into batch for faster tokenization

def tokenize_test(row):
  return tokenizer(row['test_text'], truncation = True, padding = True) #Return  tokenized text test along with other column/features (emotion labels, index)

tokenize_test_apache = lyrics_dataset['test_set'].map(tokenize_test, batched = True) #Map the function across the test dataset we have constructed, put the lyrics text into batch for faster tokenization

#Post-processing steps before feeding to the dataloader
tokenize_train_apache = tokenize_train_apache.remove_columns(["__index_level_0__", "train_text"])
tokenize_test_apache = tokenize_test_apache.remove_columns(["__index_level_0__", "test_text"])


tokenize_train_apache = tokenize_train_apache.rename_column("emotion labels", "labels") #Rename the column 'emotion labels' to 'labels' => Model expect argument 'labels'

tokenize_test_apache = tokenize_test_apache.rename_column("emotion labels", "labels")  #Rename the column 'emotion labels' to 'labels' => Model expect argument 'labels'

tokenize_train_apache.set_format("torch")
tokenize_test_apache.set_format("torch")




In [ ]:
import torch

def lab_to_int(row):
  row['labels'] = row['labels'].type(torch.LongTensor) #Casting to Long
  return row

tokenize_test_apache = tokenize_test_apache.map(lab_to_int)
print("Before:", tokenize_test_apache[2000]['labels'])
tokenize_test_apache = tokenize_test_apache.map(lab_to_int)
print("After:", tokenize_test_apache[2000]['labels'])

# Specifyin The Modeo

In [ ]:
#Load The Model and specify the class, Swap the head with 6 label classifications, specify the task to single label classification
from transformers import AutoModelForSequenceClassification
id2label = {0:"sadness", 1:"joy", 2:'love', 3:'anger', 4:'fear', 5:'surprise'}
label2id = {'sadness': 0, 'joy':1, 'love':2, 'anger': 3, 'fear': 4, 'surprise': 5}
model = AutoModelForSequenceClassification.from_pretrained(version,
                                           num_labels =6,
                                           id2label = id2label,
                                           label2id = label2id,
                                           problem_type = "single_label_classification" )

In [ ]:
#Switch to GPU
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

# Model Training And Experimental Tracking

In [ ]:
from transformers import Trainer
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
!pip install evaluate
import random
import wandb
from transformers import TrainingArguments
import evaluate

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="khuongnguyen1792005-university-of-calgary-in-alberta",
    # Set the wandb project where this run will be logged.
    project="music-sentiment-analysis-with-stopwords",
    # Track hyperparameters and run metadata.
    config={
        "dataset": "Kaggle Spotifty 500K songs with lyrics and audio features dataset (csv)",
        'data_size': 3000
        "learning_rate": 1e-5,
        "architecture": "BERT - AutoSequenceForClassification",
        "epochs": 5,
        'per_device_train_batch_size' : 32,
         'per_device_eval_batch_size' : 32,
         'optimizer' : "adamw_torch_fused",
         'lr_scheduler_type' : 'linear',
         'eval_strategy' : 'epoch',
         'logging_steps' : 23,


    },
)

training_args = TrainingArguments(
                "sentiment trainer",
                learning_rate = 1e-5,
                num_train_epochs = 5,
                per_device_train_batch_size= 32,
                per_device_eval_batch_size = 32,
                optim = "adamw_torch_fused",
                lr_scheduler_type = 'linear',
                eval_strategy = 'epoch',
                logging_steps  = 23,
                report_to = "wandb"

)

def compute_metrics(eval_pred):
  accuracy_metric = evaluate.load('accuracy')
  precision_metric = evaluate.load('precision')
  recall_metric = evaluate.load('recall')
  f1_metric = evaluate.load('f1')


  logits, labels = eval_pred

  predictions = np.argmax(logits, axis = -1)
  acc_score = accuracy_metric.compute(predictions = predictions, references = labels)
  prec_score = precision_metric.compute(predictions = predictions, references = labels, average = 'weighted')
  rec_score = recall_metric.compute(predictions = predictions, references = labels, average = 'weighted')
  f1_score = f1_metric.compute(predictions = predictions, references = labels, average = 'weighted')
  return {
      'Accuracy' : acc_score,
      'Precision': prec_score,
      'Recall': rec_score,
      'F1': f1_score,
  }

#Fine Tuning
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = tokenize_train_apache,
    eval_dataset = tokenize_test_apache,
    processing_class = tokenizer,
    compute_metrics = compute_metrics,

)

trainer.train()